# Lesson 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** သည် runtime တွင် agents များအား ကိရိယာများ၊ အရင်းအမြစ်များနှင့် ဒေတာရင်းမြစ်များကို dynamic အနေဖြင့် ရှာဖွေ၍ အသုံးပြုနိုင်စေသည့် ဖွင့်လှစ်ထားသည့် စံသတ်မှတ်ချက်တစ်ခု ဖြစ်သည်။ agent အတွင်းသို့ ကိရိယာများကို မျက်နှာတစ်ဖက်စီ hardcode ပြုလုပ်ထားရန် အစား MCP ဖြင့် agent များသည် တောင်းဆိုချက်အရ စွမ်းဆောင်ရည်များကို ထုတ်ဖော်ပြသသည့် အပြင်server များနှင့် ချိတ်ဆက်နိုင်သည်။

In this lesson, you'll learn:
- MCP ဆိုတာဘာလဲ နှင့် agent စနစ်များအတွက် အဘယ်ကြောင့် အရေးပါသနည်း
- MCP client-server ဖွဲ့စည်းပုံသည် မည်သို့ လုပ်ဆောင်သည်
- MCP ပုံစံ tool ရှာဖွေမှုကို အသုံးပြုသော agent များကို မည်သို့ တည်ဆောက်ရမည်


## တပ်ဆင်ခြင်း

**လိုအပ်ချက်များ:**
- တပ်ဆင်ထားပြီးသော မော်ဒယ်ပါရှိသည့် Azure AI Foundry ပရောဂျက်
- အတည်ပြုရန် `az login` ကို အသုံးပြုပါ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) ဆိုတာဘာလဲ?

MCP သည် AI အေဂျင့်များအတွက် ပြင်ပကိရိယာများနှင့် ဒေတာအရင်းအမြစ်များကို ရှာဖွေပြီး အပြန်အလှန် ဆက်သွယ်နိုင်ရန် စံတော်ချိန်နည်းလမ်းတစ်ခုကို သတ်မှတ်ပေးသည်။

- **MCP Server**: စံနမူနာ protocol တစ်ခုမှတဆင့် ကိရိယာများ၊ အရင်းအမြစ်များနှင့် prompts များကို ထုတ်ဖော်ပြသသည်
- **MCP Client**: ဆာဗာများနှင့် ချိတ်ဆက်ပြီး ရရှိနိုင်သည့် စွမ်းရည်များကို ရှာဖွေတွေ့ရှိပေးသော agent runtime ဖြစ်သည်
- **Dynamic Discovery**: အေဂျင့်များသည် မကြိုတင်ချပေးထားသော ကိရိယာများကို မလိုအပ်ဘဲ — runtime တွင် ရရှိနိုင်သည့် အရာများကို ရှာဖွေတွေ့ရှိနိုင်သည်

ဤကိစ္စသည် အေဂျင့်ကုဒ်ကို ပြင်ဆင်ရန် လလိုအပ်မှုမရှိဘဲ အသစ်သော စွမ်းရည်များကို ဖြည့်စွက်နိုင်သည့် ချဲ့ထွင်နိုင်သော အေဂျင့် စနစ်များကို တည်ဆောက်ရာတွင် အလွန် ပြင်းထန်သော အကျိုးရှိစေသည်။


## MCP ၏ လုပ်ဆောင်ပုံ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ကိုယ်စားလှယ် (MCP client) သည် MCP server သို့ ချိတ်ဆက်သည်
2. ဆာဗာသည် အသုံးပြုနိုင်သော ကိရိယာများနှင့် ၎င်းတို့၏ schemas ပါဝင်သည့် စာရင်းဖြင့် တုံ့ပြန်သည်
3. ကိုယ်စားလှယ်သည် ၎င်း၏ အတွေးဖြေရှင်းမှုအတွင်း တွေ့ရှိသည့် မည်သည့် ကိရိယာကိုမဆို ခေါ်နိုင်သည်
4. ရလဒ်များကို တူညီသော protocol မှတဆင့် ပြန်လည် ပေးပို့သည်


## MCP ကိရိယာ ရှာဖွေမှုကို အတုယူပြသခြင်း

အစစ် MCP ဆာဗာတစ်ခုအတွက် ပြေးနေသော ဆာဗာလုပ်ငန်းလိုအပ်သဖြင့်၊ ကျွန်တော်တို့သည် `@tool` functions များကို အသုံးပြုပြီး MCP-ချိတ်ဆက်ထားသော နေထိုင်ခွင့်ပေးသည့် ဝန်ဆောင်မှုက ပေးမယ့် အရာများကို အတုအယောင် ပြသမည်။

ထုတ်လုပ်ရေးတွင်၊ ဒီကိရိယာများကို ဒေသတွင်း locally သတ်မှတ်ထားခြင်းမဟုတ်ဘဲ MCP ဆာဗာမှ dynamic အနေဖြင့် ရှာဖွေတွေ့ရှိမည်။


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-စတိုင် ကိရိယာများဖြင့် အေးဂျင့် တည်ဆောက်ခြင်း


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ထုတ်လုပ်ရေးတွင် MCP

ထုတ်လုပ်ရေးပတ်ဝန်းကျင်တွင် MCP သည် အင်အားကြီးသော ပုံစံများကို အထောက်အပံ့ ပေးပါသည်။

 
- **ဒိုင်နမစ် ကိရိယာ ရှာဖွေခြင်း**: Agent များသည် MCP servers များနှင့် ချိတ်ဆက်ကာ runtime တွင် ကိရိယာများကို ရှာဖွေတွေ့ရှိသည်
- **ချိတ်ဆက်မှု သီးခြားထားသော ဖွဲ့စည်းပုံ**: ကိရိယာ ပံ့ပိုးသူများသည် Agent ထက် လွတ်လပ်စွာ အပ်ဒိတ်များကို ပြုလုပ်နိုင်သည်
- **အဖွဲ့အစည်း အကြား မျှဝေမှု**: အဖွဲ့များသည် MCP servers များမှတဆင့် မည်သည့် Agent မဆို အသုံးပြုနိုင်သော စွမ်းဆောင်ရည်များကို ထုတ်ဖော်နိုင်သည်
- **Microsoft Agent Framework ထောက်ပံ့မှု**: MAF သည် `mcp` ပေါင်းစည်းမှုမှတဆင့် built-in MCP client အထောက်အပံ့ရှိသည်

 
MAF နှင့် အမှန်တကယ် MCP server ကို အသုံးပြုလိုပါက `hosted_mcp_tool()` သို့မဟုတ် MCP client ပေါင်းစည်းမှုမှတဆင့် ချိတ်ဆက်ရမည်ဖြစ်သည်။

**ပိုမိုသိရှိရန်:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## အကျဉ်းချုပ်

ဒီသင်ခန်းစာတွင် သင် သိရှိခဲ့သည်:
- **MCP** သည် အေဂျင့်များနှင့် ကိရိယာပံ့ပိုးသူများအကြား ကိရိယာများကို အချိန်နှင့်တပြေးညီ ရှာဖွေဖော်ထုတ်နိုင်စေသော ဖွင့်လှစ် စံသတ်မှတ်ချက်တစ်ခုဖြစ်သည်
- ဒီ **ဖောက်သည်-ဆာဗာ ဖွဲ့စည်းမှု** သည် အေဂျင့်များကို လည်ပတ်ချိန်တွင် ၎င်းတို့၏ စွမ်းရည်များကို ရှာဖွေတွေ့ရှိနိုင်စေသည်
- MCP သည် **ချဲ့ထွင်နိုင်ပြီး ချိတ်ဆက်မှု လျော့နည်းသော အေဂျင့်စနစ်များ** ကို ကုဒ်ပြောင်းလဲမှု မလိုဘဲ ကိရိယာများ ထည့်သွင်းနိုင်စေသည်
- Microsoft Agent Framework သည် ထုတ်လုပ်မှု အသုံးပြုမှုအတွက် **ထည့်သွင်းထားသော MCP ပံ့ပိုးမှု** ကို ပံ့ပိုးပေးသည်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
သတိပေးချက်:
ဤစာရွက်ကို AI ဘာသာပြန် ဝန်ဆောင်မှုဖြစ်သည့် [Co-op Translator](https://github.com/Azure/co-op-translator)ဖြင့် ဘာသာပြန်ထားပါသည်။ တိကျမှန်ကန်မှုရရှိစေရန် ကြိုးပမ်းထားသော်လည်း အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှန်ကန်မှုလွဲယွင်းမှုများ ပါရှိနိုင်ကြောင်း သိရှိထားပါ။ မူရင်းစာရွက်ကို မူလဘာသာဖြင့် အာဏာပိုင် အရင်းအမြစ်အဖြစ် ထင်မြင်ရမည်ဖြစ်သည်။ အရေးကြီးသော အချက်အလက်များအတွက် သက်ဆိုင်ရာ ပရော်ဖက်ရှင်နယ် လူ့ဘာသာပြန်ကို အသုံးပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ကို အသုံးပြုခြင်းကြောင့် ဖြစ်ပေါ်လာနိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မှားယွင်းသော နားလည်မှုများအတွက် ကျွန်ုပ်တို့သည် တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
